# Basic NLP with `text_records()`

This notebook shows a simple handoff from `crategraph` to an NLP library.

The point of using `crategraph` here is that we do not read every file directly. We use the RO-Crate graph to select file entities, annotate those files with genre metadata from related entities, keep provenance and entity metadata with `text_records()`, then compare a couple of genre-specific subgraphs.


## Install TextBlob

Run this cell once if TextBlob is not already available in your notebook environment.

In [ ]:
!uv pip install textblob

In [ ]:
import re
from pathlib import Path

import pandas as pd
from textblob import TextBlob

from crategraph import Crate

## Load the crate

Load the Australian Corpus of English crate from `data/`.

In [ ]:
crate = Crate(Path("../../data/ldaca/Australian_Corpus_of_English"))
crate.summary()

In [ ]:
crate.glimpse()

## Annotate files with graph-derived genre

Start with file entities, then derive a `genre` property from the file-to-genre relationship recorded in the crate. The annotation is a graph transform, so the derived field behaves like any other entity property in later filtering and record export.


In [ ]:
tagged = crate.annotate_entities(
    genre=lambda entity: entity.related("ldac:linguisticGenre").join("name")
)

text_files = tagged.select(entity_types=["File"])
text_files.summary()

Make one subgraph per genre you want to compare. From here on, each analysis step can stay scoped to those graph selections.


In [ ]:
genres = ["Report", "Narrative"]

genre_graphs = {genre: text_files.where(genre=genre) for genre in genres}
{genre: len(graph.entities) for genre, graph in genre_graphs.items()}

In [ ]:
def sample_entity_ids(graph, limit=15):
    return sorted(entity.id for entity in graph.entities)[:limit]


{genre: sample_entity_ids(graph, limit=5) for genre, graph in genre_graphs.items()}

## Hand selected subgraphs to NLP tools

`text_records()` returns one row-like record per text unit, with provenance columns kept alongside the text. Because `annotate_entities()` wrote `genre` back onto the file entities, `include_properties` can carry both native metadata (`name`) and the derived genre into the same rows.


In [ ]:
def records_for_graph(graph, limit=15):
    return list(
        graph.text_records(
            include_properties=["name", "genre"],
            filters={"entity_id": sample_entity_ids(graph, limit)},
        )
    )


records_by_genre = {genre: records_for_graph(graph) for genre, graph in genre_graphs.items()}

preview_rows = [record for records in records_by_genre.values() for record in records[:3]]

pd.DataFrame(preview_rows)[["entity_id", "genre", "name", "text"]]

In [ ]:
word_pattern = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def word_count(text):
    return len(word_pattern.findall(text))


def analyse_record(record):
    return {
        **record,
        "word_count": word_count(record["text"]),
        "polarity": TextBlob(record["text"]).sentiment.polarity,
    }


analysed_by_genre = {
    genre: [analyse_record(record) for record in records]
    for genre, records in records_by_genre.items()
}

preview = [record for records in analysed_by_genre.values() for record in records[:3]]

pd.DataFrame(preview)[["entity_id", "genre", "word_count", "polarity", "name"]]

## Compare the genre subgraphs

The grouping column comes from RO-Crate relationships, not from file names. Here the comparison follows the two graph selections directly rather than regrouping the whole corpus table.


In [ ]:
def mean(values):
    values = list(values)
    return sum(values) / len(values) if values else None


def summarise_genre(genre, records):
    return {
        "genre": genre,
        "documents": len(records),
        "mean_words": mean(record["word_count"] for record in records),
        "mean_polarity": mean(record["polarity"] for record in records),
    }


summary = [summarise_genre(genre, records) for genre, records in analysed_by_genre.items()]

pd.DataFrame(summary).set_index("genre")